# Práctica · Spark DataFrames · Clientes y pedidos

Completa las celdas indicadas. Trabaja sobre el entorno dockerizado incluido en `spark_jupyter/`.

In [1]:
from iniciar_spark import get_spark
from pyspark.sql import functions as F

spark = get_spark("PracticaClientesPedidos")
print("SparkSession creada")
print("Versión de Spark:", spark.version)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/15 15:00:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession creada
Versión de Spark: 4.1.1


In [2]:
from pathlib import Path

data_dir = Path("/opt/spark-apps/datos")
clientes_path = data_dir / "clientes.csv"
pedidos_path = data_dir / "pedidos.csv"

print("Ruta clientes:", clientes_path)
print("Ruta pedidos:", pedidos_path)


Ruta clientes: /opt/spark-apps/datos/clientes.csv
Ruta pedidos: /opt/spark-apps/datos/pedidos.csv


## 1. Lee ambos CSV y muestra su esquema y algunas filas.

In [29]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Creo una sesión de Spark
spark = SparkSession.builder \
    .appName("PracticaClientesPedidos") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

# Rutas de los archivos CSV
path_clientes = "/opt/spark-apps/datos/clientes.csv"
path_pedidos = "/opt/spark-apps/datos/pedidos.csv"

# Lectura de datos

df_clientes = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ";") \
    .csv(path_clientes)

df_pedidos = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ";") \
    .csv(path_pedidos)

# Mostrar esquemas
print("--- Esquema de la tabla Clientes ---")
df_clientes.printSchema()

print("\n--- Esquema de la tabla Pedidos ---")
df_pedidos.printSchema()

# Mostrar contenido
print("\n--- Muestra de datos: Clientes ---")
df_clientes.show(5)

print("\n--- Muestra de datos: Pedidos ---")
df_pedidos.show(5)

--- Esquema de la tabla Clientes ---
root
 |-- id_cliente: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- segmento: string (nullable = true)


--- Esquema de la tabla Pedidos ---
root
 |-- id_pedido: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: double (nullable = true)
 |-- precio_unitario: integer (nullable = true)


--- Muestra de datos: Clientes ---
+----------+------+----------+--------+
|id_cliente|nombre|    ciudad|segmento|
+----------+------+----------+--------+
|         1|   Ana|  Sevilla |Estandar|
|         2|  Luis|    Bilbao| Premium|
|         3| Marta| Alicante |Estandar|
|         4| Pablo|   Madrid | Premium|
|         5| Lucia|    Bilbao| Premium|
+----------+------+----------+--------+
only showing top 5 rows

--- Muestra de datos: Pedidos ---
+---------+----------+----------+---------+--------+

## 2. Limpia el DataFrame de clientes: trim en ciudad y dropDuplicates.

In [30]:
# --- 2. Limpieza de datos ---

# A. Eliminar duplicados en clientes
df_clientes_clean = df_clientes.dropDuplicates(["id_cliente"])

# B. Eliminar espacios en columnas de texto (trim)
df_clientes_clean = df_clientes_clean \
    .withColumn("nombre", F.trim(F.col("nombre"))) \
    .withColumn("ciudad", F.trim(F.col("ciudad")))

# C. Gestionar valores nulos en clientes
df_clientes_clean = df_clientes_clean.na.fill({
    "nombre": "Sin Nombre",
    "ciudad": "Desconocida"
})

# D. Validar tipos de datos en clientes (IMPORTANTE)
df_clientes_clean = df_clientes_clean \
    .withColumn("id_cliente", F.col("id_cliente").cast("int")) \
    .withColumn("nombre", F.col("nombre").cast("string")) \
    .withColumn("ciudad", F.col("ciudad").cast("string"))

# ---------------- PEDIDOS ----------------

# Limpieza y validación de tipos en pedidos
df_pedidos_clean = df_pedidos \
    .withColumn("id_pedido", F.col("id_pedido").cast("int")) \
    .withColumn("id_cliente", F.col("id_cliente").cast("int")) \
    .withColumn("cantidad", F.col("cantidad").cast("int")) \
    .withColumn("precio_unitario", F.col("precio_unitario").cast("double")) \
    .withColumn("producto", F.trim(F.col("producto")))  # por si hay espacios

# Manejo de nulos en pedidos (opcional pero recomendable)
df_pedidos_clean = df_pedidos_clean.na.fill({
    "producto": "Sin producto"
})

# --- RESULTADOS ---
print("Clientes tras limpieza:")
df_clientes_clean.printSchema()
df_clientes_clean.show(5)

print("\nPedidos tras limpieza:")
df_pedidos_clean.printSchema()
df_pedidos_clean.show(5)

Clientes tras limpieza:
root
 |-- id_cliente: integer (nullable = true)
 |-- nombre: string (nullable = false)
 |-- ciudad: string (nullable = false)
 |-- segmento: string (nullable = true)



[Stage 6:>                                                          (0 + 1) / 1]

+----------+------+--------+--------+
|id_cliente|nombre|  ciudad|segmento|
+----------+------+--------+--------+
|         1|   Ana| Sevilla|Estandar|
|         2|  Luis|  Bilbao| Premium|
|         3| Marta|Alicante|Estandar|
|         4| Pablo|  Madrid| Premium|
|         5| Lucia|  Bilbao| Premium|
+----------+------+--------+--------+
only showing top 5 rows

Pedidos tras limpieza:
root
 |-- id_pedido: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = false)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: double (nullable = true)

+---------+----------+----------+---------+--------+---------------+
|id_pedido|id_cliente|     fecha| producto|cantidad|precio_unitario|
+---------+----------+----------+---------+--------+---------------+
|     1001|        12|2025-03-05|    Ratón|       6|           16.0|
|     1002|        32|2025-02-02|    Ratón|       3|           19.0|
|     1003

## 3. Convierte tipos en pedidos y crea la columna `importe`.

In [31]:
# TODO
# --- 3. Transformación ---
# Validamos tipos: cantidad (Int) y precio (Double)
# Creamos la columna 'importe' (cantidad * precio)
df_pedidos_clean = df_pedidos \
    .withColumn("cantidad", F.col("cantidad").cast("int")) \
    .withColumn("precio_unitario", F.col("precio_unitario").cast("double")) \
    .withColumn("importe", F.round(F.col("cantidad") * F.col("precio_unitario"), 2))

print("Pedidos con columna importe:")
df_pedidos_clean.show(5)

Pedidos con columna importe:
+---------+----------+----------+---------+--------+---------------+-------+
|id_pedido|id_cliente|     fecha| producto|cantidad|precio_unitario|importe|
+---------+----------+----------+---------+--------+---------------+-------+
|     1001|        12|2025-03-05|    Ratón|       6|           16.0|   96.0|
|     1002|        32|2025-02-02|    Ratón|       3|           19.0|   57.0|
|     1003|         5|2025-03-07|  Monitor|       2|          210.0|  420.0|
|     1004|        13|2025-03-18|Impresora|       4|          205.0|  820.0|
|     1005|        41|2025-03-10|  Teclado|       5|           40.0|  200.0|
+---------+----------+----------+---------+--------+---------------+-------+
only showing top 5 rows


## 4. Haz un join entre clientes y pedidos.

In [32]:
# --- 4. Join entre clientes y pedidos ---

# Realizamos un inner join usando la columna común id_cliente

df_join = df_clientes_clean.join(
    df_pedidos_clean,
    "id_cliente",
    "inner"
)

print(f"Total registros en Join: {df_join.count()}")
df_join.show(5)

Total registros en Join: 110
+----------+------+--------+--------+---------+----------+---------+--------+---------------+-------+
|id_cliente|nombre|  ciudad|segmento|id_pedido|     fecha| producto|cantidad|precio_unitario|importe|
+----------+------+--------+--------+---------+----------+---------+--------+---------------+-------+
|        12|Miguel|Zaragoza|Estandar|     1001|2025-03-05|    Ratón|       6|           16.0|   96.0|
|        32|Andres|Zaragoza| Premium|     1002|2025-02-02|    Ratón|       3|           19.0|   57.0|
|         5| Lucia|  Bilbao| Premium|     1003|2025-03-07|  Monitor|       2|          210.0|  420.0|
|        13| Irene|  Madrid| Premium|     1004|2025-03-18|Impresora|       4|          205.0|  820.0|
|         5| Lucia|  Bilbao| Premium|     1006|2025-03-05|  Teclado|    NULL|           35.0|   NULL|
+----------+------+--------+--------+---------+----------+---------+--------+---------------+-------+
only showing top 5 rows


## 5. Filtra ventas Premium con importe >= 100.

In [33]:
# TODO
# --- 5. Filtra ventas Premium ---
# Ventas con importe mayor o igual a 100
df_premium = df_join.filter(F.col("importe") >= 100)
print(f"Ventas Premium encontradas: {df_premium.count()}")

Ventas Premium encontradas: 83


## 6. Clasifica pedidos con `when` en Alto / Medio / Bajo.

In [34]:
# --- 6. Clasifica pedidos con 'when' ---
# Clasificación según el importe

df_clasificado = df_join.withColumn(
    "segmento_pedido",
    F.when(F.col("importe") >= 200, "Alto")
     .when((F.col("importe") >= 50) & (F.col("importe") < 200), "Medio")
     .otherwise("Bajo")
)

# Mostrar resultado
df_clasificado.select(
    "id_pedido",     # ← nombre correcto
    "importe",
    "segmento_pedido"
).show(5)

+---------+-------+---------------+
|id_pedido|importe|segmento_pedido|
+---------+-------+---------------+
|     1001|   96.0|          Medio|
|     1002|   57.0|          Medio|
|     1003|  420.0|           Alto|
|     1004|  820.0|           Alto|
|     1006|   NULL|           Bajo|
+---------+-------+---------------+
only showing top 5 rows


## 7. Calcula agregaciones por ciudad y segmento.

In [35]:
# --- 7. Agregaciones por ciudad y segmento ---

df_metricas = df_clasificado.groupBy(
    "ciudad",
    "segmento_pedido"
).agg(
    # Número de pedidos por ciudad y segmento
    F.count("id_pedido").alias("num_pedidos"),

    # Suma total de ingresos
    F.sum("importe").alias("ingresos_totales"),

    # Media del importe
    F.round(F.avg("importe"), 2).alias("media_importe")

).orderBy(
    "ciudad",
    F.desc("ingresos_totales")
)

df_metricas.show()

+----------+---------------+-----------+----------------+-------------+
|    ciudad|segmento_pedido|num_pedidos|ingresos_totales|media_importe|
+----------+---------------+-----------+----------------+-------------+
|  Alicante|           Alto|         11|         15855.0|      1441.36|
|  Alicante|          Medio|          6|           730.0|       121.67|
|    Bilbao|           Alto|         18|         16578.0|        921.0|
|    Bilbao|          Medio|          6|           717.0|        119.5|
|    Bilbao|           Bajo|          3|            64.0|         32.0|
|   Granada|           Alto|          8|          9745.0|      1218.13|
|   Granada|          Medio|          4|           544.0|        136.0|
|    Madrid|           Alto|          6|          4528.0|       754.67|
|    Madrid|          Medio|          4|           538.0|        134.5|
|    Malaga|           Bajo|          1|            34.0|         34.0|
|    Murcia|           Alto|          5|          2924.0|       

## 8. Crea una vista temporal y haz una consulta SQL.

In [36]:
# TODO
# --- 8. Vista temporal y SQL ---
df_clasificado.createOrReplaceTempView("ventas_totales")

# Consulta: Top 3 ciudades con más ingresos
query_sql = """
    SELECT ciudad, SUM(importe) as total_ventas
    FROM ventas_totales
    GROUP BY ciudad
    ORDER BY total_ventas DESC
    LIMIT 3
"""
df_sql = spark.sql(query_sql)
df_sql.show()

+--------+------------+
|  ciudad|total_ventas|
+--------+------------+
|  Bilbao|     17359.0|
|Alicante|     16585.0|
| Granada|     10289.0|
+--------+------------+



## 9. Usa `sample()` y `randomSplit()`.

In [37]:
# TODO
# --- 9. sample() y randomSplit() ---
# Tomamos una muestra del 10% sin reemplazo
df_muestra = df_clasificado.sample(withReplacement=False, fraction=0.1, seed=42)

# Dividimos el dataset (80% entrenamiento, 20% test)
train_df, test_df = df_clasificado.randomSplit([0.8, 0.2], seed=42)

print(f"Muestra: {df_muestra.count()} | Train: {train_df.count()} | Test: {test_df.count()}")


Muestra: 15 | Train: 92 | Test: 18


## 10. Guarda el resultado en Parquet en `/opt/spark-apps/salida/resultado_parquet` y léelo de nuevo.

In [38]:
# TODO
# --- 10. Persistencia ---
path_salida = "/opt/spark-apps/salida/resultado_parquet"

# Guardamos el resultado (usamos overwrite para evitar errores si ya existe)
df_clasificado.write.mode("overwrite").parquet(path_salida)

# Leemos de nuevo para verificar
df_parquet = spark.read.parquet(path_salida)
print("Datos leídos desde Parquet:")
df_parquet.printSchema()

Datos leídos desde Parquet:
root
 |-- id_cliente: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- segmento: string (nullable = true)
 |-- id_pedido: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: double (nullable = true)
 |-- importe: double (nullable = true)
 |-- segmento_pedido: string (nullable = true)



## 11. Responde brevemente en Markdown:
- ¿Qué ventaja tiene usar join?
- ¿Qué diferencia hay entre sample y randomSplit?
- ¿Qué pedido se pierde en el inner join y por qué?

Permite combinar dos DataFrames usando una clave común, evitando duplicar información y facilitando análisis más completos al unir datos relacionados.

-sample: selecciona una parte del DataFrame (porcentaje), devolviendo un único subconjunto.
-randomSplit: divide el DataFrame en varios subconjuntos distintos según proporciones.


Se pierde el pedido que no tiene coincidencia en la tabla contraria, porque el inner join solo conserva las filas donde la clave existe en ambas tablas.

## 12. Cierra la sesión de Spark.

In [21]:
spark.stop()